# Stage the live pulse crossing

Run twice: PHASE='baseline', then after the agent has observed baseline set PHASE='update'. Upload the resulting one-line JSON file through Eventhouse Get data > Local file. This notebook does NOT claim automatic Eventhouse ingestion. Use a unique RUN_ID and the same value on both runs.


In [ ]:
import sys
from pathlib import Path
BASE = Path('/lakehouse/default/Files/underwriting_demo')
sys.path.insert(0, str(BASE / 'src'))
from insurance_demo import build_data, validate, column_type, AS_OF, ROOT
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType
TABLE_PREFIX = 'demo_'

def save_rows(name, rows):
    types = {'bit': BooleanType(), 'bigint': LongType(), 'float': DoubleType()}
    schema = StructType([StructField(field, types.get(column_type(field, rows), StringType()), True) for field in rows[0]])
    frame = spark.createDataFrame(rows, schema)
    for field in rows[0]:
        datatype = column_type(field, rows)
        if datatype == 'date':
            frame = frame.withColumn(field, F.to_date(field))
        elif datatype == 'datetime2(6)':
            frame = frame.withColumn(field, F.to_timestamp(field))
    destination = TABLE_PREFIX + name
    if spark.catalog.tableExists(destination):
        properties = {row.key: row.value for row in spark.sql(f'SHOW TBLPROPERTIES {destination}').collect()}
        assert properties.get('delta.columnMapping.mode', 'none') == 'none', 'Use a new managed demo table without column mapping'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(destination)
    assert spark.table(destination).count() == len(rows)
    return frame

import json
from datetime import datetime, timezone
from insurance_demo import live_events
RUN_ID = 'rehearsal-001'
PHASE = 'baseline'
assert PHASE in ('baseline', 'update')
events = live_events(build_data(), RUN_ID, datetime.now(timezone.utc))
event = events[0 if PHASE == 'baseline' else 1]
event['EventTime'] = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
output = BASE / f'{RUN_ID}-{PHASE}.jsonl'
output.write_text(json.dumps(event) + '\n', encoding='utf-8')
print(str(output))
print(event)
